In [1]:
using Revise

In [2]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

Load schedule

In [3]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_kronecker, seed="seed123");

Reify the schedule. Need to do a dryrun to get all the primitives

In [4]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkRepresentation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)

GeneRegulatorySystems.Models.NetworkRepresentation.Entity(:v1_model, :v1_model, Dict{Symbol, Any}(:polymerases => :polymerases, :proteasomes => :proteasomes, :ribosomes => :ribosomes), GeneRegulatorySystems.Models.NetworkRepresentation.Entity[GeneRegulatorySystems.Models.NetworkRepresentation.Entity(:gene, Symbol("29"), Dict{Symbol, Any}(), GeneRegulatorySystems.Models.NetworkRepresentation.Entity[GeneRegulatorySystems.Models.NetworkRepresentation.Entity(:species, Symbol("29.proteins"), Dict{Symbol, Any}(:species_type => "proteins"), GeneRegulatorySystems.Models.NetworkRepresentation.Entity[], GeneRegulatorySystems.Models.NetworkRepresentation.Link[]), GeneRegulatorySystems.Models.NetworkRepresentation.Entity(:species, Symbol("29.active"), Dict{Symbol, Any}(:species_type => "active"), GeneRegulatorySystems.Models.NetworkRepresentation.Entity[], GeneRegulatorySystems.Models.NetworkRepresentation.Link[]), GeneRegulatorySystems.Models.NetworkRepresentation.Entity(:species, Symbol("29.elon

In [ ]:
example_dir = "examples"
schedule_files = String[]

for (root, dirs, files) in walkdir(example_dir)
    for file in files
        if endswith(file, ".schedule.json")
            push!(schedule_files, joinpath(root, file))
        end
    end
end

results = Dict()

for path in schedule_files
    try
        @info "Processing: $path"
        sched = Models.load(path, seed="seed123")
        net = extract_network(sched)
        if net !== nothing
            flat_nodes, flat_links = NetworkRepresentation.flatten(net)
            num_nodes = length(flat_nodes)
            num_links = length(flat_links)
            @info "  Success: $(num_nodes) nodes, $(num_links) links"
            results[path] = "Success (nodes=$num_nodes, links=$num_links)"
        else
            @warn "  Failed: network is nothing"
            results[path] = "Failed"
        end
    catch e
        @error "  Error: $e"
        results[path] = "Error: $(e)"
    end
end

┌ Info: Processing: examples/benchmark/differentiated-medium.schedule.json
└ @ Main /Users/2648894/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:16
┌ Info:   Success: 5303 nodes, 5805 links
└ @ Main /Users/2648894/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:23
┌ Info: Processing: examples/benchmark/differentiated-small.schedule.json
└ @ Main /Users/2648894/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:16
┌ Info:   Success: 1316 nodes, 1427 links
└ @ Main /Users/2648894/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:23
